In [4]:
import requests
import pandas as pd
import time
from tqdm import tqdm
import xml.etree.ElementTree as ET
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google.colab import files

##############################################################
# ROBUST CONNECTION SETUP (Prevents Colab Disconnects)
##############################################################
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

##############################################################
# USER SETTINGS
##############################################################

EMAIL = "simon.anguzu@students.mak.ac.ug"       # Required by NCBI
API_KEY = ""                                    # Optional but recommended

# Updated date range: 2010 to 2026
START_YEAR = "2010"
END_YEAR = "2026"

# Comprehensive search query covering bees, pesticides, and omics/expression terms
SEARCH_QUERY = r'''
("honey bee*" OR "honeybee*" OR "Apis" OR "Apis mellifera" OR "Apis cerana" OR "Apis dorsata" OR "stingless bee*" OR "Meliponini" OR "Melipona" OR "Meliponula" OR "bumblebee*" OR "Bombus" OR "Bombus terrestris" OR "Bombus impatiens" OR "non-Apis" OR "wild bee*" OR "pollinator*" OR "Apoidea")
AND
("pesticide*" OR "agrochemical*" OR "insecticide*" OR "neonicotinoid*" OR "imidacloprid" OR "thiamethoxam" OR "clothianidin" OR "acetamiprid" OR "thiacloprid" OR "flupyradifurone" OR "fungicide*" OR "herbicide*" OR "glyphosate" OR "acaricide*" OR "fipronil" OR "organophosphate*" OR "pyrethroid*" OR "carbamate*" OR "chlorpyrifos" OR "deltamethrin")
AND
("transcriptom*" OR "RNA-Seq" OR "RNA sequencing" OR "gene expression" OR "differentially expressed" OR "DEG" OR "DEGs" OR "microarray*" OR "proteom*" OR "metabolom*" OR "multi-omics" OR "multiomics" OR "ecotoxicogenom*" OR "toxicogenom*" OR "next generation sequencing" OR "NGS")
'''

##############################################################
# SEARCH PUBMED (2010–2026 Range)
##############################################################

base_search = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pubmed",
    "term": SEARCH_QUERY,
    "datetype": "pdat",       # Date type: Publication Date
    "mindate": START_YEAR,    # 2010
    "maxdate": END_YEAR,      # Includes 2026 publications
    "retmode": "json",
    "retmax": 0,              # Get count first
    "email": EMAIL
}

if API_KEY:
    params["api_key"] = API_KEY

print("Querying NCBI to find total record count...")
r = session.get(base_search, params=params)
r.raise_for_status()

data = r.json()
count = int(data["esearchresult"]["count"])

print(f"\nTotal PubMed records found ({START_YEAR}-{END_YEAR}): {count}\n")

##############################################################
# DOWNLOAD ALL PMIDs
##############################################################

pmids = []
batch_size = 500

for start in tqdm(range(0, count, batch_size), desc="Fetching PMIDs"):
    search_params = {
        "db": "pubmed",
        "term": SEARCH_QUERY,
        "datetype": "pdat",
        "mindate": START_YEAR,
        "maxdate": END_YEAR,
        "retstart": start,
        "retmax": batch_size,
        "retmode": "json",
        "email": EMAIL
    }

    if API_KEY:
        search_params["api_key"] = API_KEY

    r = session.get(base_search, params=search_params)
    ids = r.json().get("esearchresult", {}).get("idlist", [])
    pmids.extend(ids)

    time.sleep(0.35)

print(f"\nTotal PMIDs downloaded: {len(pmids)}\n")

##############################################################
# FETCH METADATA
##############################################################

fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
records = []
fetch_batch = 200

for i in tqdm(range(0, len(pmids), fetch_batch), desc="Fetching Metadata"):
    ids_chunk = ",".join(pmids[i:i+fetch_batch])

    fetch_params = {
        "db": "pubmed",
        "id": ids_chunk,
        "retmode": "xml",
        "email": EMAIL
    }

    if API_KEY:
        fetch_params["api_key"] = API_KEY

    response = session.get(fetch_url, params=fetch_params)

    try:
        root = ET.fromstring(response.content)

        for article in root.findall(".//PubmedArticle"):
            try:
                title = article.findtext(".//ArticleTitle")

                abstract_parts = [a.text for a in article.findall(".//AbstractText") if a.text]
                abstract = " ".join(abstract_parts)

                journal = article.findtext(".//Journal/Title")

                year = article.findtext(".//PubDate/Year")
                if not year:
                    year = article.findtext(".//PubDate/MedlineDate")

                pmid = article.findtext(".//PMID")

                doi = ""
                for aid in article.findall(".//ArticleId"):
                    if aid.attrib.get("IdType") == "doi":
                        doi = aid.text
                        break

                authors = []
                for author in article.findall(".//Author"):
                    last = author.findtext("LastName")
                    first = author.findtext("ForeName")
                    if last and first:
                        authors.append(f"{first} {last}")

                author_string = "; ".join(authors)
                url = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"

                records.append({
                    "PMID": pmid,
                    "DOI": doi,
                    "Title": title,
                    "Authors": author_string,
                    "Journal": journal,
                    "Year": year,
                    "Abstract": abstract,
                    "PubMed URL": url
                })
            except Exception:
                continue

    except ET.ParseError:
        print(f"\nXML parsing error for chunk. Skipping batch.")

    time.sleep(0.35)

##############################################################
# SAVE AND DOWNLOAD
##############################################################

df = pd.DataFrame(records)
output_filename = f"PubMed_Bee_Toxicogenomics_{START_YEAR}_{END_YEAR}.xlsx"
df.to_excel(output_filename, index=False, engine='openpyxl')

print("\nDone Extraction!")
print(f"Total metadata records extracted: {len(df)}")

# Download directly to your local computer from Colab
files.download(output_filename)

Querying NCBI to find total record count...

Total PubMed records found (2010-2026): 420



Fetching PMIDs: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]



Total PMIDs downloaded: 420



Fetching Metadata: 100%|██████████| 3/3 [00:06<00:00,  2.09s/it]



Done Extraction!
Total metadata records extracted: 420


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>